### Source to Raw - Ingestion

#### 1. Getting Parameter Values

In [0]:
sourceObjectName = dbutils.widgets.get("sourceObjectName").strip().lower()
sourceSystem = dbutils.widgets.get("sourceSystem").strip().lower()

#### 2. Importing Libraries

In [0]:
import os
import logging
from dateutil import parser
from datetime import datetime, timedelta
import json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import current_timestamp,to_timestamp,trim,lower,when,max,min,current_date,monotonically_increasing_id,col,  cast,hex, expr, lit,to_date
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

#### 3. Run Logging Notebook

In [0]:
%run ../utils/loggingNotebook

#### 4. Run Utils Notebook

In [0]:
%run ../utils/utilsMethods

#### 5. Running Params Notebook

In [0]:
%run ../utils/params

#### 6. Defining Logging Parameters

In [0]:
maxSequence = spark.sql(f"""select max(Cast(reverse(split(adbPipelineId, '_'))[0]AS int)) FROM {pipelineAuditTable} WHERE lower(sourceObjectName) = lower('{sourceObjectName}') and lower(sourceSystem) = lower('{sourceSystem}') and stageOrder = {rawOrderStage}""").collect()[0][0]

if maxSequence == None:
    sequence = "1"
else:
    sequence = str(maxSequence + 1)

In [0]:
try:
    #Fetching the path of the notebook
    notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    notebookName = notebookPath.split("/")[-1] #Splitting the path to get notebook name
    
    if sourceObjectName != "" and sourceSystem != "":
        if "." in sourceObjectName:
            schema = sourceObjectName.split('.')[0]
            objectName = sourceObjectName.split('.')[1]
            adbpipelineId = f"{sourceSystem}_{schema}_{objectName}_{rawStage}_{sequence}"
        else:
            schema = None
            objectName = sourceObjectName
            adbpipelineId = f"{sourceSystem}_{objectName}_{rawStage}_{sequence}"
      
    else:
        adbpipelineId = f"{sourceSystem}_{schema}_{objectName}_{rawStage}_{sequence}"
        properties={'custom_dimensions': {'adbPipelineID':f'{adbpipelineId}','notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
        logger.exception("Empty sourceObjectName/sourceSystem",extra=properties)
        raise Exception("Following field(s) is empty - sourceObjectName, sourceSystem in the Workflow Parameters")
   
  
    properties={'custom_dimensions': {'adbPipelineID':f'{adbpipelineId}','notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
    logger.info(f"Parameters {sourceSystem} and {sourceObjectName} have been retrieved successfully from the Workflow.",extra=properties)
    #Variables for Audit Table
    stageStartTs = datetime.now()
    stageObjectName = objectName
    stageName = rawStage.lower().strip()
    stageOrder = int(rawOrderStage)
    

except Exception as e:
    logger.exception(f"An error occurred while processing the input values from the Workflow : {str(e)}",extra=properties)
    raise e  


#### 8. Loading Metastore Tables -Job Config File

In [0]:
#Reading the jobConfig.csv file into a dataframe
try:
    logger.info(f"Checking if config information file exists at the {jobConfigPath}",extra=properties)
    if not dbutils.fs.ls(jobConfigPath):
        logger.Exception("Mandatory Job Configuration  file not found",extra=properties)
        raise Exception(f"Job Configuration file is not found at {jobConfigPath}.")
       
    else:
        configFolderPath=dbutils.fs.ls(jobConfigPath)
        logger.info("Reading the Job Configuration file from given location", extra = properties)
        configDf = readSparkCsv(jobConfigPath)
        configDf = configDf.select([when(trim(lower(col(c)))=="null",None).otherwise(col(c)).alias(c) for c in configDf.columns])
            
except Exception as e:
    logger.exception(f"Error while loading Job Configuration file: {str(e)}", extra = properties)
    raise e

#### 9. Loading Metastore Tables -JobControl Delta table

In [0]:
try:
    jobControlDf = spark.read.format("delta").load(jobControlPath)
    logger.info("Have Read the Job Control table into jobControlDf",extra=properties)

except Exception as e:
    logger.exception("An error occurred while loading the job control table",extra=properties)
    raise Exception("An error occurred while loading the job control table: " + str(e))


#### 10. Check - If sourceSystem & sourceObjectName Parameters are valid

In [0]:
try:
    if (sourceObjectName not in [row[0].lower().strip() for row in configDf.select("sourceObjectName").distinct().collect()]):
        #Log error message if the source Object Name is not present in configDf
        logger.exception(f"Invalid sourceObjectName parameter: {sourceObjectName} in the Workflow Parameter.Please provide a valid Source Object Name.", extra = properties)
        raise ValueError(f"Invalid sourceObjectName parameter: {sourceObjectName} in the Workflow Parameter.Please provide a valid Source Object Name.")

    # Checking if sourceSystem parameter is present in configDf
    if sourceSystem not in [row[0].lower() for row in (configDf.select("sourceSystem")).distinct().collect()]:
        
        #Log error message if the sourceSystem is not present in configDf
        logger.exception(f"Invalid source System parameter: {sourceSystem} in the Workflow Parameter.Please provide a valid Source System Name.", extra = properties)
        raise ValueError(f"Invalid sourceSystem parameter: {sourceSystem} in the Workflow Parameter.Please provide a valid Source System Name.")
        
    logger.info(f"Parameters {sourceObjectName} and {sourceSystem} validated successfully from the Config File.", extra = properties)
        
except Exception as e:
    logger.exception(f"Error while validating parameters from the Config File,Please provide a Valid Source Object Name and Source System Name in the Workflow.: {str(e)}", extra = properties)
    raise e

#### 11. Filtering  the Config file  based on the Source System and Source Object Name and Checking for Duplicate and Not null records for the Mandatory Columns

In [0]:
#Reading specific record of sourceObjectName and storing into Dictionary
try:
    logger.info(f"Filtering config details for {sourceObjectName} and {sourceSystem} from the config file.",extra = properties)
    configRows = configDf.filter((trim(lower(configDf.sourceObjectName))==(sourceObjectName)) & (trim(lower(configDf.sourceSystem))==(sourceSystem)))
    #Checking for Duplicates
    if configRows.count() > configRows.dropDuplicates(["sourceSystem","sourceObjectName"]).count():
        logger.exception(f"Duplicate Entry found in Job Config File for {sourceObjectName} and {sourceSystem}.",extra = properties)
        raise ValueError(f"Duplicate Entry found in Job Config File for {sourceObjectName} and {sourceSystem}.")
    else:
        logger.info(f"No Duplicate Entry found in Job Config File for {sourceObjectName} and {sourceSystem}.",extra = properties)
        
        #Creating a dict on the filtered out data 
        config=configRows.first().asDict()
        #Checking for mandatory columns to be not null
        if config['rawObjectFormat'].lower().strip()=="table":
            if config['workloadType'].lower().strip()=="custom":
                missingCols = [col for col in mandatoryColsSourceCustomTable if col not in configRows.columns]
                if missingCols:
                    logger.exception(f"Mandatory column(s) {missingCols} is/are missing from the config information file.",extra = properties)
                    raise ValueError(f"Mandatory column(s) {missingCols} is/are missing from the config information file.")
                else:
                    logger.info(f"Mandatory column(s) {mandatoryColsSourceCustomTable} are present in config information file.",extra=properties)
                    # Null records in the Mandatory columns check
                    for column in mandatoryColsSourceCustomTable:
                        # Filtering the dataframe and creating a new df with records having null values
                        nullCount = configRows.where(f"{column} is NULL").count()
                        if nullCount > 0:
                            
                            logger.exception(f"Found {nullCount} null values in the column {column}.", extra = properties)
                            raise ValueError(f"{nullCount} null values found in the mandatory column {column}. Job terminated.")

                        logger.info(f"No null records found in {mandatoryColsSourceCustomTable}",extra = properties) 
            else: 
                missingCols = [col for col in mandatoryColsSourceTable if col not in configRows.columns]
                if missingCols:
                    logger.exception(f"Mandatory column(s) {missingCols} is/are missing from the config information file.",extra = properties)
                    raise ValueError(f"Mandatory column(s) {missingCols} is/are missing from the config information file.")
                else:
                    logger.info(f"Mandatory column(s) {mandatoryColsSourceTable} are present in config information file.",extra=properties)
                    
                    for column in mandatoryColsSourceTable:
                        
                        nullCount = configRows.where(f"{column} is NULL").count()
                        if nullCount > 0:
                            
                            logger.exception(f"Found {nullCount} null values in the column {column}.", extra = properties)
                            raise ValueError(f"{nullCount} null values found in the mandatory column {column}. Job terminated.")

                        logger.info(f"No null records found in {mandatoryColsSourceTable}",extra = properties) 

                
        elif config['rawObjectFormat'].lower().strip()=="csv":
            if config['workloadType'].lower().strip()=="custom":
                missingCols = [col for col in mandatoryColsSourceCustomCsv if col not in configRows.columns]
                if missingCols:
                    logger.exception(f"Mandatory column(s) {missingCols} is/are missing from the config information file.",extra = properties)
                    raise ValueError(f"Mandatory column(s) {missingCols} is/are missing from the config information file.")
                else:
                    logger.info(f"Mandatory column(s) {mandatoryColsSourceCustomCsv} are present in config information file.",extra=properties)
                    # Null records in the Mandatory columns check
                    for column in mandatoryColsSourceCustomCsv:
                        # Filtering the dataframe and creating a new df with records having null values
                        nullCount = configRows.where(f"{column} is NULL").count()
                        if nullCount > 0:
                            
                            logger.exception(f"Found {nullCount} null values in the column {column}.", extra = properties)
                            raise ValueError(f"{nullCount} null values found in the mandatory column {column}. Job terminated.")

                        logger.info(f"No null records found in {mandatoryColsSourceCustomCsv}",extra = properties) 
            else: 
                missingCols = [col for col in mandatoryColsSourceCsv if col not in configRows.columns]
                if missingCols:
                    logger.exception(f"Mandatory column(s) {missingCols} is/are missing from the config information file.",extra = properties)
                    raise ValueError(f"Mandatory column(s) {missingCols} is/are missing from the config information file.")
                else:
                    logger.info(f"Mandatory column(s) {mandatoryColsSourceCsv} are present in config information file.",extra=properties)
                    # Null records in the Mandatory columns check
                    for column in mandatoryColsSourceCsv:
                        # Filtering the dataframe and creating a new df with records having null values
                        nullCount = configRows.where(f"{column} is NULL").count()
                        if nullCount > 0:
                            
                            logger.exception(f"Found {nullCount} null values in the column {column}.", extra = properties)
                            raise ValueError(f"{nullCount} null values found in the mandatory column {column}. Job terminated.")

                        logger.info(f"No null records found in {mandatoryColsSourceCsv}",extra = properties) 

        else:
            logger.exception(f"Invalid Raw Object Format provided {config['rawObjectFormat']}", extra = properties)
            raise ValueError(f"Invalid Raw Object Format provided {config['rawObjectFormat']}")

except Exception as e:
    
    #Updating pipelineAudit Table
    workloadType=configRows.select('workLoadType').first()[0]
    updateAuditTable(rawFailure,str(e))
    
    
    logger.exception(f"Error Occured while filtering the config file for {sourceObjectName} and {sourceSystem}: {str(e)}", extra = properties)
    raise e 


#### 12. Reading the Job Control Table based on the Source System and Source Object Name

In [0]:
#Reading specific record of sourceSystem from JC_config_df for lastRunDate and currentRunDate and storing into Dictionary

jcConfig = jobControlDf.filter((trim(lower(jobControlDf.sourceSystem))) == (sourceSystem)).first().asDict()
logger.info(f"Reading the job control details for {sourceObjectName} and {sourceSystem} from the Job Control table.",extra = properties)

#### 13. Validating if Active Flag is Y 

In [0]:
try:
    activeFlag= [d for d in config['activeFlag']]
    for flags in activeFlag:
        if flags.lower().strip()=='y':
            logger.info(f"Active Flag in Job Config File is 'Y',Proceed with the next step",extra = properties)
        elif flags.lower().strip()=='n':
            logger.info(f"Active Flag in Job Config File is 'N',Cannot proceed with the next step",extra = properties)
            dbutils.notebook.exit(f"Notebook {notebookName} exited as the Active Flag is N for {config['sourceObjectName']},Cannot proceed with the Loads")        
        else:
            logger.exception(f"Active Flag in Job Config File is {flags},Cannot proceed with the next step",extra = properties)
            raise ValueError(f"Active Flag in Job Config File is  {flags},Cannot proceed with the next step")
        
except Exception as e:
    #Updating pipelineAudit Table
    workloadType=config['workloadType']
    
    logger.exception(f"Active Flag in Job Config File is {flags},Cannot proceed with the next process", extra = properties)
    raise e         

In [0]:
from pyspark.sql.functions import col
import datetime
try:
    adhocRunTableDf=spark.sql(f"select * from {adhocRunTable}")
    if not adhocRunTableDf.filter((trim(lower(adhocRunTableDf.sourceObjectName))==(sourceObjectName)) & (trim(lower(adhocRunTableDf.sourceSystem))==(sourceSystem))).count()>0:
        InsertAdhocQuery = f"""INSERT INTO {adhocRunTable} (sourceSystem, sourceObjectName, adhocRunFlag) VALUES ('{sourceSystem}','{sourceObjectName}','N')"""
        spark.sql(InsertAdhocQuery)
        logger.info("Inserted a new entry in the Adhoc Run Table",extra = properties)
    elif  adhocRunTableDf.filter((trim(lower(adhocRunTableDf.sourceObjectName))==(sourceObjectName)) & (trim(lower(adhocRunTableDf.sourceSystem))==(sourceSystem))).count()==1:
        logger.info("Entries already exists in the Adhoc Run Table",extra=properties)
    else:
        logger.exception("Duplicate Entries found in the Adhoc Run Table",extra=properties)
        raise ValueError("Duplicate Entries found in the Adhoc Run Table")
        
except Exception as e:   
    logger.exception(f"Inserting a new entry in adhocRunTable has failed,for the new source system and source object Combination: {str(e)}",extra=properties)
    raise e                

#### 14. Connections

In [0]:
%run ../utils/connections

#### 15. Validating the Source Object Type

In [0]:
try:
    if config['sourceType'].lower().strip()==sourceType:
         logger.info(f"Valid Source Type for the {sourceObjectName} table",extra = properties)
    else:
        logger.exception(f"Invalid Source Type for the {sourceObjectName} table",extra = properties)
        raise ValueError(f"Invalid Source Type for the {sourceObjectName} table")
        
except Exception as e:
    
    workloadType=config['workloadType']
    updateAuditTable(rawFailure,str(e))
    
    logger.exception(f"Invalid Source Type for the {sourceObjectName} table", extra = properties)
    raise e         


#### 16. Initiating the Process Based on the Raw Object Format

In [0]:
try:
    if config['rawObjectFormat'].lower().strip() == "table":
        
               
        #Archival Process
        parquetPath = (f"""{baseAdlsPath}{config["parquetPath"]}""")
        archivalParquetPath =(f"""{baseAdlsPath}{config["archivalParquetPath"]}""")
        tempPath=(f"""{baseAdlsPath}{config["archivalParquetPath"]}/tempArchival/""")
      
        logger.info(f"""Full parquetPath is:{parquetPath}""",extra = properties)
        logger.info(f"""Full archivalParquetPath is:{archivalParquetPath}""",extra = properties)
        logger.info(f"""Temporary Archival Path is :{tempPath}""",extra = properties)
        if filesCheck(parquetPath) == False:
            initialLoad="True"
            logger.info("There is no directory with sourceObjectName - {} , you can start with Ingestion".format(config['sourceObjectName'].lower().capitalize()),extra=properties)
                
        else:
            
            TempArchivalProcess(config,parquetPath,tempPath,'parquet')
                   
        
        #Calling the Respective Processes to Ingest the Data From Source To Raw

        logger.info(f"Calling respective method based on the Workload Type to read the {sourceObjectName} from source",extra=properties)
        if initialLoad=="True" or config["workloadType"].lower().strip()=='historical':
            if config['customConfigJson']==None:
                tableDf, countSource=readSqlserverHistorical(config,jcConfig)
            else:
                tableDf, countSource=readSqlserverHistoricalOptimized(config,jcConfig)
            if initialLoad=='True' and countSource==0:
                logger.info("Ingestion did not happen for this Load as the data read from source was 0",extra = properties)
                dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
            elif countSource!=0:
                tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
                if initialLoad=='False':
                    lstParquet=[c[0] for c in dbutils.fs.ls(parquetPath) if c[0].split("/")[-1].endswith(".parquet")]
                    if len(lstParquet)>0 and  countSource==tableDfOutCount:
                        ingestionStatus=True
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
        
                
                    else:
                        ingestionStatus=False
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                elif initialLoad=='True' and countSource!=tableDfOutCount:
                    logger.exception(f"The Source and Target Count Does not match after Ingestion",extra=properties)
                    raise ValueError(f"The Source and Target Count Does not match after Ingestion")     
                
                
                  
            elif initialLoad=='True'  and countSource==tableDfOutCount:
                logger.info(f"The Data is Ingested successfully with the matching counts",extra=properties)
            elif initialLoad=='False' and countSource==0:
                ingestionStatus='Empty'
                FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
            
                   
                    
        elif config["workloadType"].lower().strip()=='histincr':
            lstParquet=[c[0] for c in dbutils.fs.ls(tempPath) if c[0].split("/")[-1].endswith(".parquet")]
                
            if len(lstParquet)>0:
                tableDf, countSource=readSqlserverIncremental(config, jcConfig)
                if initialLoad=='True' and countSource==0:
                    logger.info("Ingestion did not happen for this Load as the data read from source was 0",extra = properties)
                    dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
                elif countSource!=0:
                    tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
                    if initialLoad=='False':
                        lstParquet=[c[0] for c in dbutils.fs.ls(parquetPath) if c[0].split("/")[-1].endswith(".parquet")]
                        if len(lstParquet)>0  and countSource==tableDfOutCount:
                            ingestionStatus=True
                            FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
        
                
                        else:
                            ingestionStatus=False
                            FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                    elif initialLoad=='True' and countSource!=tableDfOutCount:
                        logger.exception(f"The Source and Target Count Does not match after Ingestion",extra=properties)
                        raise ValueError(f"The Source and Target Count Does not match after Ingestion")     
                
                
                  
                elif initialLoad=='True'  and countSource==tableDfOutCount:
                    logger.info(f"The Data is Ingested successfully with the matching counts",extra=properties)
                elif initialLoad=='False' and countSource==0:
                    ingestionStatus='Empty'
                    FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
            else:
                #tableDf, countSource=readSqlserverHistorical(config,jcConfig)
                if config['customConfigJson']==None:
                    tableDf, countSource=readSqlserverHistorical(config,jcConfig)
                else:
                    tableDf, countSource=readSqlserverHistoricalOptimized(config,jcConfig)                
                if initialLoad=='True' and countSource==0:
                    logger.info("Ingestion did not happen for this Load as the data read from source was 0",extra = properties)
                    dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
                elif countSource!=0:
                    tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
                    if initialLoad=='False':
                        lstParquet=[c[0] for c in dbutils.fs.ls(parquetPath) if c[0].split("/")[-1].endswith(".parquet")]
                        if len(lstParquet)>0  and countSource==tableDfOutCount:
                            ingestionStatus=True
                            FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
        
                
                        else:
                            ingestionStatus=False
                            FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                    elif initialLoad=='True' and countSource!=tableDfOutCount:
                        logger.exception(f"The Source and Target Count Does not match after Ingestion",extra=properties)
                        raise ValueError(f"The Source and Target Count Does not match after Ingestion")     
                
                
                  
                elif initialLoad=='True'  and countSource==tableDfOutCount:
                    logger.info(f"The Data is Ingested successfully with the matching counts",extra=properties)
                elif initialLoad=='False' and countSource==0:
                    ingestionStatus='Empty'
                    FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                    
            
                       
        elif config["workloadType"].lower().strip() == "incremental":
            
            tableDf, countSource=readSqlserverIncremental(config, jcConfig)
            if initialLoad=='True' and countSource==0:
                    logger.info("Ingestion did not happen for this Load as the data read from source was 0",extra = properties)
                    dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
            elif countSource!=0:
                tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
                if initialLoad=='False':
                    lstParquet=[c[0] for c in dbutils.fs.ls(parquetPath) if c[0].split("/")[-1].endswith(".parquet")]
                    if len(lstParquet)>0  and countSource==tableDfOutCount:
                        ingestionStatus=True
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
        
                
                    else:
                        ingestionStatus=False
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                elif initialLoad=='True' and countSource!=tableDfOutCount:
                    logger.exception(f"The Source and Target Count Does not match after Ingestion",extra=properties)
                    raise ValueError(f"The Source and Target Count Does not match after Ingestion")     
            
                
                  
            elif initialLoad=='True'  and countSource==tableDfOutCount:
                logger.info(f"The Data is Ingested successfully with the matching counts",extra=properties)
            elif initialLoad=='False' and countSource==0:
                ingestionStatus='Empty'
                FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
            
                    
        elif config["workloadType"].lower().strip() == "custom":
            jcConfig["lastRunDate"] = parser.parse(config["loadStartTime"])
            jcConfig["currentRunDate"] = parser.parse(config["loadEndTime"])
            logger.info("Using lastRunDate as: "+ jcConfig["lastRunDate"].strftime("%Y-%m-%d'T'%H:%M:%S"),extra=properties)
            logger.info("Using currentRunDate as: "+ jcConfig["currentRunDate"].strftime("%Y-%m-%d'T'%H:%M:%S"),extra=properties)
            tableDf, countSource=readSqlserverCustom(config, jcConfig)
            tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
    
            if initialLoad=='True' and countSource==0:
                logger.info("Ingestion did not happen for this Load as the data read from source was 0",extra = properties)
                dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
            elif countSource!=0:
                tableDfOut,tableDfOutCount=dataIngestionProcess(tableDf)
                if initialLoad=='False':
                    lstParquet=[c[0] for c in dbutils.fs.ls(parquetPath) if c[0].split("/")[-1].endswith(".parquet")]
                    if len(lstParquet)>0  and countSource==tableDfOutCount:
                        ingestionStatus=True
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
    
            
                    else:
                        ingestionStatus=False
                        FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                elif initialLoad=='True' and countSource!=tableDfOutCount:
                    logger.exception(f"The Source and Target Count Does not match after Ingestion",extra=properties)
                    raise ValueError(f"The Source and Target Count Does not match after Ingestion")     
            
            
              
            elif initialLoad=='True'  and countSource==tableDfOutCount:
                logger.info(f"The Data is Ingested successfully with the matching counts",extra=properties)
            elif initialLoad=='False' and countSource==0:
                ingestionStatus='Empty'
                FinalArchivalProcess(tempPath,archivalParquetPath,parquetPath,ingestionStatus)
                    
                            
        else:
            
            logger.exception(f"Invalid workloadType: {config['workloadType']}",extra=properties)
            raise ValueError(f"Invalid workloadType: {config['workloadType']}")
        
        if config['dateColumnName']!=None:
            srcStartTs=tableDf.select(min(config['dateColumnName'])).first()[0]
            srcEndTs=tableDf.select(max(config['dateColumnName'])).first()[0]
        else:
            srcStartTs="null"
            srcEndTs="null"
        #Updating Pipeline Audit if the Data Ingestion Succeeeds
        
        workloadType=config['workloadType']
        updateAuditTable(rawSuccess,"")
          
            
    elif config['rawObjectFormat'].lower().strip() == "csv":
        
               
        #Archival Process
        sourceFormatPath = (f"""{baseAdlsPath}{config['sourceFormatPath']}""")
        archivalSourceFormatPath = (f"""{baseAdlsPath}{config['archivalSourceFormatPath']}""")
        tempPath=(f"""{baseAdlsPath}{config['archivalSourceFormatPath']}/tempArchival/""")
      
        logger.info(f"""Full sourceFormatPath is:{sourceFormatPath}""",extra = properties)
        logger.info(f"""Full archivalSourceFormatPath is:{archivalSourceFormatPath}""",extra = properties)
        logger.info(f"""Temporary Archival Path is :{tempPath}""",extra = properties)
        
        if filesCheck(parquetPath) == False:
            initialLoad="True"
            logger.info("There is no directory with sourceObjectName - {} , you can start with Ingestion".format(config['sourceObjectName'].lower().capitalize()),extra=properties)
                
        else:
            TempArchivalProcess(config,parquetPath,tempPath,'csv')
        
        #Calling the Respective Processes to Ingest the Data From Source To Raw

        # Read data from the source
        logger.info(f"Calling respective method based on the Workload Type to read the {sourceObjectName} from source",extra=properties)
        if initialLoad or config["workloadType"].lower().strip()=='historical':
            tableDf, countSource=readAdlsHistorical(config,jcConfig)
            logger.info("Calling generateMd5 method to add dwmd5CheckSum column ",extra=properties)
            tableDfChcksum = generateMd5(tableDf)
            # If 'isDeleted' column exists, set the 'cdcIndicator' column to 'A' for rows where 'isDeleted' is '0' and 'D' for other rows
            if 'isDeleted' in tableDfChcksum.columns:
                logger.info("Adding audit column cdcIndicator",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", when(tableDfChcksum['isDeleted'] == '0','A').otherwise('D'))
            else:
                # If 'isDeleted' column doesn't exist, set the 'cdcIndicator' column to 'A' for all rows
                logger.info("Adding audit column cdcIndicator with 'A'",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", lit('A'))
                # Add the current timestamp as 'dwModifiedDate' column
            logger.info("Adding dwModifiedDate column with current timestamp",extra=properties)
            tableDfWrite = tableDfFinal.withColumn("dwModifiedDate", current_timestamp())
            logger.info(f"tableDf - dataframe, which has read the {sourceObjectName} from source Location",extra=properties)

            logger.info(f"Initializing process of writing tableDf into {config['rawObjectFormat']} at its path",extra=properties)
            srcStartTs=tableDfWrite.select(min('dateLastSync')).first()[0]
            srcEndTs=tableDfWrite.select(max('dateLastSync')).first()[0]

            tableDfWrite.write.format(config['rawObjectFormat']).option("inferSchema", True).mode("overwrite").path(f"{sourceFormatPath}/")
            if initialLoad=="False":
                if 'csv' in (f.name for f in dbutils.fs.ls(sourceFormatPath)):
                    ingestionStatus="success"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)
                else:
                    ingestionStatus="failed"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)

        elif config["workloadType"].lower().strip() == "incremental":
            tableDf, countSource=readAdlsIncremental(config, jcConfig)
            logger.info("Calling generateMd5 method to add dwmd5CheckSum column ",extra=properties)
            tableDfChcksum = generateMd5(tableDf)
            # If 'isDeleted' column exists, set the 'cdcIndicator' column to 'A' for rows where 'isDeleted' is '0' and 'D' for other rows
            if 'isDeleted' in tableDfChcksum.columns:
                logger.info("Adding audit column cdcIndicator",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", when(tableDfChcksum['isDeleted'] == '0','A').otherwise('D'))
            else:
                # If 'isDeleted' column doesn't exist, set the 'cdcIndicator' column to 'A' for all rows
                logger.info("Adding audit column cdcIndicator with 'A'",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", lit('A'))
                # Add the current timestamp as 'dwModifiedDate' column
            logger.info("Adding dwModifiedDate column with current timestamp",extra=properties)
            tableDfWrite = tableDfFinal.withColumn("dwModifiedDate", current_timestamp())
            logger.info(f"tableDf - dataframe, which has read the {sourceObjectName} from source Location",extra=properties)

            logger.info(f"Initializing process of writing tableDf into {config['rawObjectFormat']} at its path",extra=properties)
            srcStartTs=tableDfWrite.select(min('dateLastSync')).first()[0]
            srcEndTs=tableDfWrite.select(max('dateLastSync')).first()[0]

            tableDfWrite.write.format(config['rawObjectFormat']).option("inferSchema", True).mode("overwrite").path(f"{sourceFormatPath}/")
            if initialLoad=="False":
                if 'csv' in (f.name for f in dbutils.fs.ls(sourceFormatPath)):
                    ingestionStatus="success"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)
                else:
                    ingestionStatus="failed"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)

        elif config["workloadType"].lower().strip() == "custom":
            jcConfig["lastRunDate"] = config["loadStartTime"]
            jcConfig["currentRunDate"] = config["loadEndTime"]
            logger.info("Using lastRunDate as: "+ jcConfig["lastRunDate"].strftime("%Y-%m-%d'T'%H:%M:%S"),extra=properties)
            logger.info("Using currentRunDate as: "+ jcConfig["currentRunDate"].strftime("%Y-%m-%d'T'%H:%M:%S"),extra=properties)
            tableDf, countSource=readAdlsCustom(config, jcConfig)
            logger.info("Calling generateMd5 method to add dwmd5CheckSum column ",extra=properties)
            tableDfChcksum = generateMd5(tableDf)
            # If 'isDeleted' column exists, set the 'cdcIndicator' column to 'A' for rows where 'isDeleted' is '0' and 'D' for other rows
            if 'isDeleted' in tableDfChcksum.columns:
                logger.info("Adding audit column cdcIndicator",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", when(tableDfChcksum['isDeleted'] == '0','A').otherwise('D'))
            else:
                # If 'isDeleted' column doesn't exist, set the 'cdcIndicator' column to 'A' for all rows
                logger.info("Adding audit column cdcIndicator with 'A'",extra=properties)
                tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", lit('A'))
                # Add the current timestamp as 'dwModifiedDate' column
            logger.info("Adding dwModifiedDate column with current timestamp",extra=properties)
            tableDfWrite = tableDfFinal.withColumn("dwModifiedDate", current_timestamp())
            logger.info(f"tableDf - dataframe, which has read the {sourceObjectName} from source Location",extra=properties)
            logger.info(f"Initializing process of writing tableDf into {config['rawObjectFormat']} at its path",extra=properties)
            srcStartTs=tableDfWrite.select(min('dateLastSync')).first()[0]
            srcEndTs=tableDfWrite.select(max('dateLastSync')).first()[0]

            tableDfWrite.write.format(config['rawObjectFormat']).option("inferSchema", True).mode("overwrite").path(f"{sourceFormatPath}/") 
            if initialLoad=="False":
                if 'csv' in (f.name for f in dbutils.fs.ls(sourceFormatPath)):
                    ingestionStatus="success"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)
                else:
                    ingestionStatus="failed"
                    FinalArchivalProcess(tempPath,archivalSourceFormatPath,sourceFormatPath,ingestionStatus)

        else:

            # Raise an exception if the value of "workloadType" is not "custom" or "incremental"
            logger.exception(f"Invalid workloadType: {config['workloadType']}",extra=properties)
            raise ValueError(f"Invalid workloadType: {config['workloadType']}")

        #Updating Pipeline Audit if the Data Ingestion Succeeeds
        srcStartTs=tableDfWrite.select(config['dateColumnName']).first()[0]
        srcEndTs=tableDfWrite.select(config['dateColumnName']).first()[0]
        workloadType=config['workloadType']
        updateAuditTable(rawSuccess,"")
            
    else:
        
        logger.exception(f"Raw Object Format for {sourceObjectName} is none of these values table/csv, as of now. Check and change in the Config File",extra=properties)
        raise ValueError(f"Raw Object Format for {sourceObjectName} is none of these values table/csv, as of now. Check and change in the Config File")

except Exception as e:
    
    #Updating pipelineAudit Table
    workloadType=config['workloadType']
    updateAuditTable(rawFailure,str(e))
     
    #Log and raise error
    logger.exception(f"Error while Ingesting the data for {sourceObjectName} table", extra = properties)
    raise e         
                    